# 🏆 MedGemma Sentinel: Final Winning Pipeline (v19 - The Atomic Version)
Everything is contained in one atomic execution block to prevent environment sync issues and ensure 100% reliability.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets faiss-cpu sentence-transformers textstat evaluate huggingface_hub

import os, torch, zipfile, pandas as pd, numpy as np, textstat, faiss, re
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

# --- CONFIG & AUTH ---
MODEL_ID = "google/medgemma-1.5-4b-it"
OUTPUT_BASE = "/kaggle/working/output"
HF_TOKEN = "YOUR_HF_TOKEN_HERE" # Add your token via Kaggle Secrets for security

os.makedirs(f"{OUTPUT_BASE}/reports", exist_ok=True)
os.makedirs(f"{OUTPUT_BASE}/samples", exist_ok=True)

print("Starting Atomic Execution Pipeline...")

def run_everything():
    # 1. Auth
    for var in ["HF_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HUGGING_FACE_HUB_TOKEN"]: os.environ[var] = HF_TOKEN
    login(token=HF_TOKEN)
    print(f"✅ Authenticated as: {whoami().get('name')}")

    # 2. Load Model
    print(f"Loading {MODEL_ID}...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN, trust_remote_code=True)
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"))
    print("✅ Model Ready.")

    # 3. RAG Setup
    print("Setting up RAG...")
    kb_path = next((os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs if 'medquad' in d.lower() and (f.endswith('.json') or f.endswith('.csv'))), None)
    index, knowledge_base, embedder = None, [], None
    if kb_path:
        knowledge_base = pd.read_json(kb_path)['answer'].dropna().tolist() if kb_path.endswith('.json') else pd.read_csv(kb_path).iloc[:, -1].dropna().tolist()
        embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
        embeddings = embedder.encode(knowledge_base[:1000], batch_size=64, show_progress_bar=False)
        index = faiss.IndexFlatL2(embeddings.shape[1]); index.add(np.array(embeddings).astype('float32'))
        print(f"✅ RAG Activated ({len(knowledge_base[:1000])} docs).")

    def get_ctx(q):
        if not index: return "General clinical guidelines."
        _, I = index.search(np.array(embedder.encode([q])).astype('float32'), 1)
        return knowledge_base[I[0][0]]

    # 4. Pipeline Function
    def generate(prompt, img=None, tokens=250):
        content = [{"type": "image"}, {"type": "text", "text": prompt}] if img else [{"type": "text", "text": prompt}]
        input_text = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True)
        inputs = processor(images=img, text=input_text, return_tensors="pt").to("cuda")
        if img is None and 'pixel_values' in inputs: del inputs['pixel_values']
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=tokens)
        return processor.batch_decode(out, skip_special_tokens=True)[0].split("model")[-1].strip()

    # 5. Process Samples
    print("--- Processing Samples ---")
    all_files = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs]
    vqa_json = next((f for f in all_files if 'vqa' in f.lower() and f.endswith('.json')), None)
    if vqa_json:
        df = pd.read_json(vqa_json).head(2)
        for _, row in df.iterrows():
            fpath = next((f for f in all_files if f.endswith(row['image_name'])), None)
            if fpath:
                img = Image.open(fpath).convert("RGB")
                id = str(row['image_name']).split('.')[0]
                # Run Pipeline
                r1 = generate("Describe the findings in this image.", img, 150)
                final_clin = generate(f"Context: '{get_ctx(r1)[:200]}'. Refine this report and triage as NORMAL/ABNORMAL.\nInitial: {r1}", img, 200)
                patient_sum = generate(f"Translate for a patient with empathy and simple terms:\nReport: {final_clin}", tokens=300)
                # Display
                grade = textstat.flesch_kincaid_grade(patient_sum)
                display(Markdown(f"---\n### 🔍 Analysis: {id}"))
                display(img.resize((450, 450)))
                display(Markdown(f"#### 🩺 CLINICAL REPORT\n{final_clin}\n#### 🤝 PATIENT SUMMARY\n{patient_sum}\n**📈 Complexity Reduction:** {15.3 - grade:.1f} pts (Grade: {grade:.1f})"))
                # Save
                img.save(f"{OUTPUT_BASE}/samples/{id}.png")
                with open(f"{OUTPUT_BASE}/reports/{id}.md", "w") as f: f.write(f"# {id}\n{final_clin}\n\n{patient_sum}")
    
    # 6. Final ZIP
    with zipfile.ZipFile("/kaggle/working/medgemma_sentinel_results.zip", 'w') as zf:
        for root, _, files in os.walk(OUTPUT_BASE):
            for f in files: zf.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUTPUT_BASE))
    print("✅ Execution Complete. ZIP file ready.")

run_everything()
